# Devanagari Handwritten Character Recognition — Fixed Training Notebook

## Key fixes over original:
- **Augmentation**: RandomRotation, RandomAffine, RandomPerspective on train set
- **Normalization**: `Normalize([0.5], [0.5])` matches `app.py` preprocessing exactly
- **Separate transforms** for train vs val/test (no augmentation at eval time)
- **Better architecture**: BatchNorm + proper spatial dims + two FC layers
- **Adam + CosineAnnealingLR** instead of SGD + inverse LR
- **Label smoothing** to reduce overconfidence
- **Saves best checkpoint** by val accuracy, not last epoch
- **State dict save** (not full object) for reliable loading in app.py

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from collections import Counter
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Dataset paths
TRAIN_PATH = "/kaggle/input/datasets/suyashnpande/devanagarihandwrittencharacterdataset/DevanagariHandwrittenCharacterDataset/Train"
TEST_PATH  = "/kaggle/input/datasets/suyashnpande/devanagarihandwrittencharacterdataset/DevanagariHandwrittenCharacterDataset/Test"

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("Imports done.")

In [ ]:
# ── Device ──
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ── Inspect dataset structure ──
print("=== TRAIN CLASSES ===")
train_classes = sorted(os.listdir(TRAIN_PATH))
for c in train_classes[:10]:
    n = len(os.listdir(os.path.join(TRAIN_PATH, c)))
    print(f"  {c}: {n} images")
print(f"  ... ({len(train_classes)} total classes)")

print("\n=== TEST CLASSES ===")
test_classes = sorted(os.listdir(TEST_PATH))
for c in test_classes[:5]:
    n = len(os.listdir(os.path.join(TEST_PATH, c)))
    print(f"  {c}: {n} images")
print(f"  ... ({len(test_classes)} total classes)")

In [ ]:
# ── FIX 1: Separate transforms for train vs val/test ──
# FIX 2: Normalize([0.5],[0.5]) to match app.py preprocess_image exactly
# FIX 3: Data augmentation so model generalises to hand-drawn input

train_transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((32, 32)),
    transforms.RandomRotation(10),                                     # ±10° rotation
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1),           # shift up to 10%
                            scale=(0.85, 1.15)),                        # zoom ±15%
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),         # mild perspective warp
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),                                 # → [-1, 1]
])

# NO augmentation at eval time — only resize + normalize
eval_transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

print("Transforms defined.")
print("Train transform:", train_transform)
print("\nEval transform:", eval_transform)

In [ ]:
# ── Dataset loading with correct split ──
# Load train folder twice: once with train_transform, once with eval_transform
# This ensures val set sees eval_transform (no augmentation), not train_transform

train_full_aug  = datasets.ImageFolder(TRAIN_PATH, transform=train_transform)
train_full_eval = datasets.ImageFolder(TRAIN_PATH, transform=eval_transform)
test_dataset    = datasets.ImageFolder(TEST_PATH,  transform=eval_transform)

# 80/20 split with fixed seed
n_total    = len(train_full_aug)
n_train    = int(0.8 * n_total)
n_val      = n_total - n_train

generator  = torch.Generator().manual_seed(SEED)
train_idx, val_idx = random_split(range(n_total), [n_train, n_val], generator=generator)

train_dataset = Subset(train_full_aug,  train_idx.indices)   # augmented
val_dataset   = Subset(train_full_eval, val_idx.indices)     # clean

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

# idx_to_class from train_full_aug (same folder order)
idx_to_class = {v: k for k, v in train_full_aug.class_to_idx.items()}

print(f"Train samples      : {len(train_dataset)}")
print(f"Validation samples : {len(val_dataset)}")
print(f"Test samples       : {len(test_dataset)}")
print(f"Num classes        : {len(train_full_aug.classes)}")
print(f"\nClass→idx (first 5): {dict(list(train_full_aug.class_to_idx.items())[:5])}")

In [ ]:
# ── Sanity check: batch shapes and pixel value range ──
for X, y in train_loader:
    print(f"Train batch — images: {X.shape}, labels: {y.shape}")
    print(f"Pixel range  — min: {X.min():.3f}, max: {X.max():.3f}, mean: {X.mean():.3f}")
    print("Expected: min≈-1.0, max≈1.0  (confirms Normalize([0.5],[0.5]) is applied)")
    break

In [ ]:
# ── Visualise sample images per class ──
varnamala_order = [
    "character_1_ka",  "character_2_kha",  "character_3_ga",   "character_4_gha",
    "character_5_kna", "character_6_cha",  "character_7_chha", "character_8_ja",
    "character_9_jha", "character_10_yna", "character_11_taamatar", "character_12_thaa",
    "character_13_daa","character_14_dhaa","character_15_adna","character_16_tabala",
    "character_17_tha","character_18_da",  "character_19_dha", "character_20_na",
    "character_21_pa", "character_22_pha", "character_23_ba",  "character_24_bha",
    "character_25_ma", "character_26_yaw", "character_27_ra",  "character_28_la",
    "character_29_waw","character_30_motosaw","character_31_petchiryakha",
    "character_32_patalosaw","character_33_ha","character_34_chhya",
    "character_35_tra","character_36_gya",
    "digit_0","digit_1","digit_2","digit_3","digit_4",
    "digit_5","digit_6","digit_7","digit_8","digit_9"
]

# Collect one raw (un-normalised) sample per class for display
raw_transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])
raw_ds = datasets.ImageFolder(TRAIN_PATH, transform=raw_transform)
raw_idx_to_class = {v: k for k, v in raw_ds.class_to_idx.items()}

samples = {}
for img, label in raw_ds:
    cname = raw_idx_to_class[label]
    if cname not in samples:
        samples[cname] = img
    if len(samples) == len(raw_ds.classes):
        break

cols = 6
rows = (len(varnamala_order) + cols - 1) // cols
plt.figure(figsize=(14, 14))
for i, cname in enumerate(varnamala_order):
    img = samples[cname].squeeze(0)
    plt.subplot(rows, cols, i + 1)
    plt.imshow(img, cmap="gray")
    plt.title(cname.replace("character_","").replace("digit_","d"), fontsize=7)
    plt.axis("off")
plt.suptitle("All 46 Classes — Devanagari Dataset", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Class balance check ──
label_counts = Counter(train_full_aug.targets)
print(f"{'Class':<35} {'Count':>6}")
print("-" * 43)
for label_idx, count in sorted(label_counts.items()):
    print(f"{idx_to_class[label_idx]:<35} {count:>6}")

In [ ]:
# ── FIX 4: Better model architecture ──
# Original ModelA collapsed spatial dims to 1×1 via 3× unpadded 5×5 convs
# This new version:
#   - Uses padding=2 on first conv to preserve 32×32
#   - Adds BatchNorm after every conv (faster convergence, better generalisation)
#   - Maintains spatial info through 3 pooling stages: 32→16→8→4
#   - Two FC layers with dropout for capacity and regularisation

class ModelA(nn.Module):
    def __init__(self, num_classes=46):
        super(ModelA, self).__init__()

        # Block 1: 32×32 → 16×16
        self.conv1 = nn.Conv2d(1, 64, kernel_size=5, stride=1, padding=2)
        self.bn1   = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(2, stride=2)

        # Block 2: 16×16 → 8×8
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn2   = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d(2, stride=2)

        # Block 3: 8×8 → 4×4
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.bn3   = nn.BatchNorm2d(256)
        self.pool3 = nn.MaxPool2d(2, stride=2)

        # Classifier: 256×4×4 = 4096 features
        self.dropout1 = nn.Dropout(0.5)
        self.fc1      = nn.Linear(256 * 4 * 4, 512)
        self.dropout2 = nn.Dropout(0.3)
        self.fc2      = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)          # flatten: [B, 256*4*4]
        x = self.dropout1(F.relu(self.fc1(x)))
        x = self.dropout2(x)
        return self.fc2(x)


# Sanity check
dummy = torch.zeros(1, 1, 32, 32)
m = ModelA(num_classes=46)
out = m(dummy)
print("Output shape:", out.shape)   # expect torch.Size([1, 46])
n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

In [ ]:
# ── Evaluation helper ──
def evaluate(model, loader, device):
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total

In [ ]:
# ── FIX 5: Better training loop ──
# - Adam instead of SGD (faster convergence)
# - CosineAnnealingLR (smooth LR decay)
# - Label smoothing=0.1 (reduces overconfidence / overfitting)
# - Saves BEST checkpoint by val accuracy, not last epoch

def train_model(model, train_loader, val_loader, epochs=30, lr=0.001, device="cpu",
                save_path="/kaggle/working/dhcd_modelA_best.pt"):

    model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    best_val_acc   = 0.0
    train_losses   = []
    val_accuracies = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        scheduler.step()
        val_acc = evaluate(model, val_loader, device=device)
        train_losses.append(running_loss)
        val_accuracies.append(val_acc)

        tag = ""
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                "model_state" : model.state_dict(),
                "num_classes" : 46,
                "class_to_idx": train_full_aug.class_to_idx,
                "epoch"       : epoch + 1,
                "val_acc"     : val_acc,
            }, save_path)
            tag = "  ← best saved"

        print(f"Epoch {epoch+1:>2}/{epochs} | "
              f"Loss: {running_loss:.4f} | "
              f"Val Acc: {val_acc:.4f} | "
              f"LR: {scheduler.get_last_lr()[0]:.6f}{tag}")

    print(f"\nBest Val Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")
    return model, train_losses, val_accuracies

In [ ]:
# ── Train ──
model = ModelA(num_classes=46)
trained_model, train_losses, val_accuracies = train_model(
    model,
    train_loader,
    val_loader,
    epochs=30,
    lr=0.001,
    device=device,
    save_path="/kaggle/working/dhcd_modelA_best.pt"
)

In [ ]:
# ── Training curves ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, len(train_losses)+1), train_losses, color="royalblue", marker="o", markersize=4)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Total Loss")
ax1.set_title("Training Loss vs Epochs")
ax1.grid(True)

ax2.plot(range(1, len(val_accuracies)+1), val_accuracies, color="orange", marker="o", markersize=4)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Validation Accuracy")
ax2.set_title("Validation Accuracy vs Epochs")
ax2.set_ylim(0, 1)
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# ── Load BEST checkpoint (not last epoch) for evaluation ──
best_ckpt = torch.load("/kaggle/working/dhcd_modelA_best.pt", map_location=device)
best_model = ModelA(num_classes=46)
best_model.load_state_dict(best_ckpt["model_state"])
best_model = best_model.to(device)
best_model.eval()

print(f"Loaded best checkpoint from epoch {best_ckpt['epoch']} "
      f"(val_acc={best_ckpt['val_acc']:.4f})")

In [ ]:
# ── Final accuracy on all splits ──
train_acc = evaluate(best_model, train_loader, device=device)
val_acc   = evaluate(best_model, val_loader,   device=device)
test_acc  = evaluate(best_model, test_loader,  device=device)

print(f"Train Accuracy      : {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"Validation Accuracy : {val_acc:.4f}  ({val_acc*100:.2f}%)")
print(f"Test Accuracy       : {test_acc:.4f}  ({test_acc*100:.2f}%)  ← final score")
print()
if train_acc - test_acc > 0.05:
    print("⚠  Gap > 5% — some overfitting remains. Consider more epochs or stronger augmentation.")
else:
    print("✓  Train/test gap is small — model generalises well.")

In [ ]:
# ── Per-class classification report ──
def get_all_preds(model, loader, device):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, pred = outputs.max(1)
            preds.extend(pred.cpu().numpy())
            trues.extend(labels.cpu().numpy())
    return np.array(trues), np.array(preds)

y_true, y_pred = get_all_preds(best_model, test_loader, device)
class_names    = [idx_to_class[i] for i in range(len(train_full_aug.classes))]

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# ── Confusion matrix ──
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=False, cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.title("Confusion Matrix — Devanagari ModelA (best checkpoint)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(rotation=90, fontsize=6)
plt.yticks(rotation=0,  fontsize=6)
plt.tight_layout()
plt.show()

In [ ]:
# ── Spot-check random test samples ──
import random

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("Random test samples: True vs Predicted", fontsize=13)
indices = random.sample(range(len(test_dataset)), 10)

best_model.eval()
for ax, idx in zip(axes.flatten(), indices):
    img, label = test_dataset[idx]
    with torch.no_grad():
        out  = best_model(img.unsqueeze(0).to(device))
        pred = out.argmax(1).item()

    # Denormalize for display: [-1,1] → [0,1]
    display_img = img.squeeze().numpy() * 0.5 + 0.5
    ax.imshow(display_img, cmap="gray")

    true_name = idx_to_class[label].replace("character_","").replace("digit_","d")
    pred_name = idx_to_class[pred].replace("character_","").replace("digit_","d")
    color = "green" if pred == label else "red"
    ax.set_title(f"T:{true_name}\nP:{pred_name}", fontsize=8, color=color)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ── FIX 6: Save final model as state_dict (not full object) ──
# This is what app.py expects when loading with:
#   ckpt = torch.load(path)
#   model.load_state_dict(ckpt["model_state"])

FINAL_SAVE_PATH = "/kaggle/working/devanagari_model.pt"

torch.save({
    "model_state" : best_model.state_dict(),
    "num_classes" : 46,
    "class_to_idx": train_full_aug.class_to_idx,
    "val_acc"     : best_ckpt["val_acc"],
    "epoch"       : best_ckpt["epoch"],
    "img_size"    : 32,
    "normalize"   : {"mean": [0.5], "std": [0.5]},   # reminder for app.py
}, FINAL_SAVE_PATH)

print(f"Saved to: {FINAL_SAVE_PATH}")

# Verify it reloads cleanly
verify_ckpt  = torch.load(FINAL_SAVE_PATH, map_location="cpu")
verify_model = ModelA(num_classes=46)
verify_model.load_state_dict(verify_ckpt["model_state"])
verify_model.eval()

dummy = torch.zeros(1, 1, 32, 32)
with torch.no_grad():
    out = verify_model(dummy)
print(f"Reload verification — output shape: {out.shape}  ✓")
print(f"Checkpoint keys: {list(verify_ckpt.keys())}")

## app.py loading snippet

Use this exact code in `app.py` to load the saved model:

```python
class ModelA(nn.Module):
    def __init__(self, num_classes=46):
        super(ModelA, self).__init__()
        self.conv1 = nn.Conv2d(1, 64, kernel_size=5, stride=1, padding=2)
        self.bn1   = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(2, stride=2)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn2   = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d(2, stride=2)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.bn3   = nn.BatchNorm2d(256)
        self.pool3 = nn.MaxPool2d(2, stride=2)
        self.dropout1 = nn.Dropout(0.5)
        self.fc1      = nn.Linear(256 * 4 * 4, 512)
        self.dropout2 = nn.Dropout(0.3)
        self.fc2      = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout1(F.relu(self.fc1(x)))
        x = self.dropout2(x)
        return self.fc2(x)

# Loading:
ckpt = torch.load(dev_path, map_location=DEVICE, weights_only=False)
dev_model = ModelA(num_classes=ckpt.get("num_classes", 46))
dev_model.load_state_dict(ckpt["model_state"])
dev_model.eval()
```